# Bag-of-Words SFT

Run supervised fine-tuning on the synthetic bag-of-words regression task using the shared BoW study config/state.

In [1]:
from pathlib import Path
import os

import polars as pl
import torch

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    os.chdir(repo_root.parent)
    repo_root = Path.cwd()

from src.experiments.bag_of_words.sft import BagOfWordsSFTConfig

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
device

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/sft.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


device(type='cuda', index=1)

## Configure

In [ ]:
config = BagOfWordsSFTConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sft-example",
    snr=0.01,
    num_words=15,
    num_samples=50_000,
    aux_words_ratio=0.5,
    prompt_length=128,
    filter_samples_above_n_tokens=384,
    word_decay_power=1.0,
    batch_size=64,
    eval_batch_size_multiple=4,
    lr_per_token=1.25e-7,
    backbone_lr_divisor=5.0,
    pad_to_multiple=8,
    train_epochs=12,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset R^2 target: {config.data.rsq:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-sft-example/15-words_snr-1.0_len-128_pow-1.0_ar-0.5
Dataset R^2 target: 0.5000
Backbone lr: 2.048e-04
Head lr:     1.024e-03


Max token length (train): 128
Max token length (val):   128
ceil_padded_seqlen:       128


Tokenize train:   0%|          | 0/49 [00:00<?, ?it/s]

Tokenize val:   0%|          | 0/49 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
state.run_training()

sft epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/195 [00:00<?, ?it/s]

epoch  0  train_corr_target=0.5743  train_corr_ground_truth=0.8186  pred_norm=5.3933  val_corr_target=0.6499  val_corr_ground_truth=0.9189


sft epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

## Results

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()